# E-Commerce Customer Insights Engine
**GetIntern.in | Week 2 Milestone Project | Merging Tables & Calculating Business Metrics**

This notebook joins customer and order data, calculates business KPIs, and builds a Top 5 customer leaderboard using **Pandas**.

**Objectives**
1. Relational table joins with `pd.merge()`
2. Metric aggregation and segmentation with `.groupby()` and `.agg()`
3. Cohort filtering and leaderboard formulation

**Business rule:** only orders with status `completed` count as purchases.

## Setup
### Step 1: Import libraries
Pandas handles the tables and NumPy supports numeric operations.

In [1]:
import pandas as pd
import numpy as np

### Step 2: Load the data
Load `users.csv` (customers) and `orders.csv`. The column `user_id` is renamed to `customer_id`, the key used to join the tables.

In [2]:
customers = pd.read_csv("customers.csv").rename(columns={"user_id": "customer_id"})
orders = pd.read_csv("orders.csv").rename(columns={"user_id": "customer_id"})

print("Customers:", customers.shape)
print("Orders:", orders.shape)
display(customers.head())
display(orders.head())

Customers: (10000, 6)
Orders: (20000, 5)


,customer_id,name,email,gender,city,signup_date
0,U000001,Angel Hill,donaldgarcia@example.net,Other,New Roberttown,2025-03-13
1,U000002,Jesse Guzman,jennifermiles@example.com,Male,South Bridget,2024-03-05
2,U000003,Adam Shaffer,jpeterson@example.org,Male,Curtisfurt,2025-07-07
3,U000004,Melanie Munoz,blairamanda@example.com,Other,New Kellystad,2024-03-07
4,U000005,Janet Williams,kendragalloway@example.org,Female,South Joshuastad,2025-01-29


,order_id,customer_id,order_date,order_status,total_amount
0,O00000001,U009310,2025-09-09T14:52:37.292731,processing,689.66
1,O00000002,U003247,2025-04-15T01:18:27.193404,completed,1666.85
2,O00000003,U007252,2025-04-27T15:37:48.008624,processing,665.06
3,O00000004,U008986,2025-10-04T20:35:22.204857,cancelled,689.50
4,O00000005,U008537,2024-11-13T08:15:18.498252,cancelled,860.50


### Step 3: Quick data quality check
Check for null values, duplicates and the distribution of `order_status`. There are no nulls or duplicates in either table. Orders are spread across five statuses: shipped, returned, completed, cancelled and processing.

In [3]:
print(customers.isnull().sum(), "\n")
print(orders.isnull().sum(), "\n")
print("Duplicate customers:", customers.duplicated().sum())
print("Duplicate orders:", orders.duplicated().sum())
print(orders["order_status"].value_counts())

customer_id    0
name           0
email          0
gender         0
city           0
signup_date    0
dtype: int64 

order_id        0
customer_id     0
order_date      0
order_status    0
total_amount    0
dtype: int64 

Duplicate customers: 0
Duplicate orders: 0
order_status
shipped       4113
returned      4066
completed     4021
cancelled     3920
processing    3880
Name: count, dtype: int64


### Step 4: Light cleaning and defining valid purchases
Remove any duplicates, strip whitespace and lowercase text columns, and convert `total_amount` to numeric. Only **completed** orders (4,021 of 20,000) are counted as real purchases, since cancelled, returned, shipped and processing orders are not confirmed revenue.

In [4]:
customers = customers.drop_duplicates()
orders = orders.drop_duplicates()

for col in ["name", "email", "gender", "city"]:
    customers[col] = customers[col].astype(str).str.strip().str.lower()
orders["order_status"] = orders["order_status"].astype(str).str.strip().str.lower()
orders["total_amount"] = pd.to_numeric(orders["total_amount"], errors="coerce")

# Only completed orders count as revenue
valid_orders = orders[orders["order_status"] == "completed"].copy()
print("All orders:", len(orders), "| Completed orders:", len(valid_orders))

All orders: 20000 | Completed orders: 4021


## Objective 1: Relational Table Joins
### Step 5: Check the join key
Confirm that `customer_id` is unique in the customers table, that every order matches a customer, and count customers with no completed orders (6,680).

In [5]:
print("Unique customers:", customers["customer_id"].nunique(), "of", len(customers))
print("Order customer_ids missing from customers:", (~valid_orders["customer_id"].isin(customers["customer_id"])).sum())
print("Customers with no completed orders:", (~customers["customer_id"].isin(valid_orders["customer_id"])).sum())

Unique customers: 10000 of 10000
Order customer_ids missing from customers: 0
Customers with no completed orders: 6680


### Step 6: Inner join
`pd.merge(..., how="inner")` keeps only customers who have at least one completed order. Result: 4,021 rows.

In [6]:
inner_df = pd.merge(customers, valid_orders, on="customer_id", how="inner")
print("Inner join shape:", inner_df.shape)
inner_df.head()

Inner join shape: (4021, 10)


,customer_id,name,email,gender,city,signup_date,order_id,order_date,order_status,total_amount
0,U000003,adam shaffer,jpeterson@example.org,male,curtisfurt,2025-07-07,O00001959,2025-09-12T21:03:13.266186,completed,1018.29
1,U000003,adam shaffer,jpeterson@example.org,male,curtisfurt,2025-07-07,O00004533,2025-10-21T08:48:45.661172,completed,108.44
2,U000006,michael santiago,lynchgeorge@example.net,male,east steven,2024-04-21,O00015666,2024-04-12T10:21:14.290876,completed,455.92
3,U000010,brandon rodriguez,laurahenderson@example.org,male,jamesmouth,2024-03-03,O00001103,2025-04-30T14:57:04.585866,completed,247.20
4,U000016,debra kim,omartinez@example.org,female,andrewside,2025-11-11,O00004418,2024-12-16T04:46:56.846948,completed,954.86


### Step 7: Left join
`pd.merge(..., how="left")` keeps every customer. Customers with no completed orders get blank order columns. Result: 10,701 rows.

In [7]:
left_df = pd.merge(customers, valid_orders, on="customer_id", how="left")
print("Left join shape:", left_df.shape)
left_df.head()

Left join shape: (10701, 10)


,customer_id,name,email,gender,city,signup_date,order_id,order_date,order_status,total_amount
0,U000001,angel hill,donaldgarcia@example.net,other,new roberttown,2025-03-13,NaN,NaN,NaN,NaN
1,U000002,jesse guzman,jennifermiles@example.com,male,south bridget,2024-03-05,NaN,NaN,NaN,NaN
2,U000003,adam shaffer,jpeterson@example.org,male,curtisfurt,2025-07-07,O00001959,2025-09-12T21:03:13.266186,completed,1018.29
3,U000003,adam shaffer,jpeterson@example.org,male,curtisfurt,2025-07-07,O00004533,2025-10-21T08:48:45.661172,completed,108.44
4,U000004,melanie munoz,blairamanda@example.com,other,new kellystad,2024-03-07,NaN,NaN,NaN,NaN


### Step 8: Compare the two joins
The left join has more rows (10,701) than the inner join (4,021) because it includes the 6,680 customers who never purchased. It also has more rows than the 10,000 customers, because customers with several orders appear once per order.

In [8]:
print("Inner join rows:", len(inner_df))
print("Left join rows:", len(left_df))
print("Customers with no orders (blank order columns):", left_df["order_id"].isna().sum())

Inner join rows: 4021
Left join rows: 10701
Customers with no orders (blank order columns): 6680


## Objective 2: Metric Aggregation & Segmentation
### Step 9: Customer-level KPIs
`.groupby("customer_id").agg()` calculates, for each customer:
- **Total spend**
- **Number of orders**
- **Average order value (AOV)**

In [9]:
customer_metrics = inner_df.groupby("customer_id").agg(
    total_spend=("total_amount", "sum"),
    num_orders=("order_id", "count"),
    avg_order_value=("total_amount", "mean"),
).reset_index()
customer_metrics.head()

,customer_id,total_spend,num_orders,avg_order_value
0,U000003,1126.73,2,563.365
1,U000006,455.92,1,455.920
2,U000010,247.20,1,247.200
3,U000016,954.86,1,954.860
4,U000019,303.73,1,303.730


### Step 10: Segmentation by city
Group by `city` to get total orders, total revenue, AOV and unique customers per city, sorted by revenue. City names are nearly unique in this dataset, so most cities have only a few customers.

In [10]:
city_summary = inner_df.groupby("city").agg(
    total_orders=("order_id", "count"),
    total_revenue=("total_amount", "sum"),
    avg_order_value=("total_amount", "mean"),
    unique_customers=("customer_id", "nunique"),
).sort_values("total_revenue", ascending=False).reset_index().round(2)
city_summary.head(10)

,city,total_orders,total_revenue,avg_order_value,unique_customers
0,north michael,8,7799.89,974.99,5
1,lindatown,3,6416.83,2138.94,2
2,west michaelville,1,6360.11,6360.11,1
3,south robertstad,2,6341.39,3170.70,1
4,turnermouth,3,6285.53,2095.18,2
5,south peterstad,1,6195.80,6195.80,1
6,west johnton,2,6027.52,3013.76,2
7,west tonyashire,2,5952.53,2976.26,1
8,port james,5,5913.48,1182.70,3
9,annettebury,3,5853.01,1951.00,1


### Revenue reconciliation check
Total revenue from the city summary equals total spend from the customer summary (2,419,712.58), confirming no revenue was lost or double-counted during the joins and aggregations.

In [11]:
print(round(city_summary["total_revenue"].sum(), 2), round(customer_metrics["total_spend"].sum(), 2))

2419712.58 2419712.58


## Objective 3: Cohort Filtering & Leaderboard
### Step 11: Identify non-converting customers
Merge the customer metrics back onto all customers with a left join. Customers with no order count (`NaN`) made zero purchases. There are 6,680 of them.

In [12]:
customer_summary = pd.merge(customers, customer_metrics, on="customer_id", how="left")
print("Customers with zero purchases:", customer_summary["num_orders"].isna().sum())

Customers with zero purchases: 6680


### Step 12: Filter out non-converting accounts
Keep only customers with at least one purchase. This reduces 10,000 customers to 3,320.

In [13]:
converting = customer_summary[customer_summary["num_orders"].notna()].copy()
print("Before:", len(customer_summary), "| After:", len(converting))

Before: 10000 | After: 3320


### Step 13: Top 5 highest-spending customers
Sort the converting customers by total spend and keep the top 5. This is the leaderboard.

In [14]:
top5 = (converting.sort_values("total_spend", ascending=False).head(5)
        [["customer_id", "name", "city", "num_orders", "total_spend", "avg_order_value"]]
        .round(2).reset_index(drop=True))
top5["name"] = top5["name"].str.title()
top5["city"] = top5["city"].str.title()
top5.index = top5.index + 1
top5

,customer_id,name,city,num_orders,total_spend,avg_order_value
1,U002814,Patricia Olson,West Michaelville,1.0,6360.11,6360.11
2,U006469,Audrey Ware,South Robertstad,2.0,6341.39,3170.70
3,U005921,Stanley Wood,South Peterstad,1.0,6195.80,6195.80
4,U005803,Alyssa Jones,Turnermouth,1.0,6066.85,6066.85
5,U003637,Eric Hernandez,West Tonyashire,2.0,5952.53,2976.26


### Step 14: Save the outputs
Export the customer summary, city summary and Top 5 table as CSV files.

In [15]:
customer_summary.to_csv("customer_summary.csv", index=False)
city_summary.to_csv("city_summary.csv", index=False)
top5.to_csv("top5_customers.csv")

### Step 15: Markdown version of the Top 5 table


In [16]:
print(top5.reset_index().rename(columns={"index": "rank"}).to_markdown(index=False))

|   rank | customer_id   | name           | city              |   num_orders |   total_spend |   avg_order_value |
|-------:|:--------------|:---------------|:------------------|-------------:|--------------:|------------------:|
|      1 | U002814       | Patricia Olson | West Michaelville |            1 |       6360.11 |           6360.11 |
|      2 | U006469       | Audrey Ware    | South Robertstad  |            2 |       6341.39 |           3170.7  |
|      3 | U005921       | Stanley Wood   | South Peterstad   |            1 |       6195.8  |           6195.8  |
|      4 | U005803       | Alyssa Jones   | Turnermouth       |            1 |       6066.85 |           6066.85 |
|      5 | U003637       | Eric Hernandez | West Tonyashire   |            2 |       5952.53 |           2976.26 |


## Conclusion
- Joined customers and orders on `customer_id` (inner join: 4,021 rows, left join: 10,701 rows).
- Calculated total spend, order count and AOV per customer, and orders, revenue and AOV per city.
- Found **6,680 customers with zero purchases** and filtered them out, leaving 3,320.
- Built the Top 5 leaderboard; the highest spender is customer U002814 with 6,360.11.